# Предобработка

### Загрузка библиотек

In [ ]:
import numpy as np
import pandas as pd 
import geopandas as gpd
import psycopg2
import phik 
from datetime import datetime 
import math
from pathlib import Path
from config.settings import DB_CONFIG

# Координаты Кремля - используем как центр
CENTER_LAT, CENTER_LON = 55.7520, 37.6175

### Загрузка датасета

In [ ]:
# данные из витрины marts.ml_listings_wide (слои raw -> stg -> dds -> marts, dbt)
conn = psycopg2.connect(**DB_CONFIG)
df = pd.read_sql_query('SELECT * FROM marts.ml_listings_wide', conn)
conn.close()
print('До удаления "мусорных:"', df.shape)

# "мусорные" - активные объявления, в нашей задаче не нужны
df = df[df['event_closed'] == 1]
print('После удаления "мусорных:"', df.shape)

In [ ]:
bool_map = {'t': True, 'f': False, True: True, False: False}
for col in ['is_apartments', 'is_new_building', 'phone_protected', 'is_studio', 'mortgage_allowed', 'nearest_metro_walk', 'demolished_in_renovation', 'is_penthouse', 'seller_is_owner']:
    df[col] = df[col].map(bool_map).astype('boolean')

df['publication_date'] = pd.to_datetime(df['publication_date'], errors='coerce')
df['first_seen'] = pd.to_datetime(df['first_seen'], errors='coerce')

# фильтр на невалидные значения - число дней на рынке, площадь, цена - должны быть больше 0
filter_days = df['days_on_market'] >= 0
filter_area = df['total_area'] > 0 
filter_price = df['price'] > 0
df = df[filter_days & filter_area & filter_price]

# формирование признака расстояния до центра Москвы
R = 6371.0 # радиус Земли в километрах
lat = np.radians(df['lat'])
lon = np.radians(df['lon'])
dlat = lat - math.radians(CENTER_LAT)
dlon = lon - math.radians(CENTER_LON)

a = (
    np.sin(dlat / 2) ** 2
    + np.cos(lat) * math.cos(math.radians(CENTER_LAT)) 
    * np.sin(dlon / 2) ** 2
)
# в километрах
df['dist_to_center'] = 2 * R * np.arcsin(np.sqrt(a))

# у объявлений без метро ближайшая станция приходит пустой - заменяем на unknown
df['nearest_metro'] = df['nearest_metro'].fillna('unknown')

# создание дополнительного признака - цена за кв.м.
df['price_per_m2'] = df['price_last'] / df['total_area']

raions = gpd.read_file('./notebooks/data/moscow_raions.geojson')[['name', 'geometry']]
raions['name'] = (
    raions['name']
    .str.replace(r'^(р-н|район)\s+', '', regex=True)
    .str.replace(r'\s+район$', '', regex=True)
    .str.strip()
)
points = gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df['lon'], df['lat']), crs=raions.crs)
joined = gpd.sjoin(points, raions, how='left', predicate='within')
joined = joined[~joined.index.duplicated(keep='first')]
df['district'] = df['district'].fillna(joined['name'])

### Устранение аномалий

In [4]:
# обрезаем значения по 1му и 99му перцентилям, чтобы избавиться от сильных выбросов
QUANT_TRESHOLD = 0.01
clip_cols = ['price', 'price_last', 'price_first', 'price_per_m2', 'total_area', 'living_area', 'kitchen_area']

for col in clip_cols:
    floor = df[col].quantile(QUANT_TRESHOLD)
    ceil = df[col].quantile(1 - QUANT_TRESHOLD)
    df = df[df[col].between(floor, ceil) | df[col].isna()]

df.to_csv('./notebooks/data/listings_preprocessed.csv', index=False)